In [ ]:
import hashlib

class BloomFilterDemo:
    def __init__(self, size=15, num_hashes=2):
        """
        size - размер битового массива (делаем маленьким, чтобы поместился на экран)
        num_hashes - количество хеш-функций
        """
        self.size = size
        self.num_hashes = num_hashes
        self.bit_array = [0] * size
        print(f"=== Инициализация Фильтра Блума ===")
        print(f"Размер массива: {size} бит")
        print(f"Количество хеш-функций: {num_hashes}\n")

    def _get_hashes(self, item):
        """Внутренняя функция: превращает строку в список индексов (от 0 до size-1)"""
        indices = []
        for i in range(self.num_hashes):
            # Солируем строку, чтобы получить разные хеши (например: "Alex:0", "Alex:1")
            salted_item = f"{item}:{i}"
            # Получаем MD5 и берем остаток от деления на размер массива
            hash_int = int(hashlib.md5(salted_item.encode()).hexdigest(), 16)
            indices.append(hash_int % self.size)
        return indices

    def add(self, item):
        """Заносит элемент в фильтр (ставит единички)"""
        indices = self._get_hashes(item)
        for idx in indices:
            self.bit_array[idx] = 1
        
        print(f"[ЗАПИСЬ] Добавляем '{item}'. Хеши: {indices}")
        print(f"Массив: {self.bit_array}\n")

    def check(self, item):
        """Проверяет наличие элемента"""
        indices = self._get_hashes(item)
        
        print(f"[ПОИСК] Ищем '{item}'. Ожидаем единички на позициях: {indices}")
        for idx in indices:
            bit_value = self.bit_array[idx]
            print(f" -> Проверяем бит {idx}: {bit_value}")
            if bit_value == 0:
                print(f" -> РЕЗУЛЬТАТ: Бит равен 0! Элемента '{item}' ТОЧНО НЕТ.")
                return False
                
        print(f" -> РЕЗУЛЬТАТ: Все нужные биты равны 1! Элемент '{item}', ВОЗМОЖНО, ЕСТЬ.")
        return True


# ========================================================
# ДЕМОНСТРАЦИОННЫЙ СЦЕНАРИЙ
# ========================================================
if __name__ == "__main__":
    # Создаем маленький фильтр, чтобы быстро его "засорить" единицами
    filter = BloomFilterDemo(size=14, num_hashes=2)
    
    # 1. Заполняем фильтр реальными пользователями
    filter.add("Alex")
    filter.add("Bob")
    filter.add("Charlie")
    
    print("="*40)
    
    # 2. Проверяем того, кто ТОЧНО ЕСТЬ (Ожидаем True)
    filter.check("Alex")
    
    print("-" * 40)
    
    # 3. Проверяем того, кого ТОЧНО НЕТ (Ожидаем False)
    # Зная алгоритм, слово "Dave" попадет на нули
    filter.check("Dave")
    
    print("\n" + "="*40)
    print("ВНИМАНИЕ: ДЕМОНСТРАЦИЯ ЛОЖНОГО СРАБАТЫВАНИЯ (False Positive)")
    print("="*40)
    
    # 4. Ищем коллизию. Мы не добавляли слово "Ivan" (и еще кучу других).
    # Но мы будем проверять случайные имена, пока биты случайно не совпадут!
    test_names = ["Eve", "Frank", "Grace", "Heidi", "Ivan", "Judy", "Kevin"]
    
    for name in test_names:
        print(f"\nТестируем '{name}':")
        # Если класс вернул True, хотя мы не делали add() - это ОНО!
        if filter.check(name):
            print(f"\n🚨 БАМ! ЛОЖНОЕ СРАБАТЫВАНИЕ! 🚨")
            print(f"Мы НИКОГДА не добавляли '{name}' в базу!")
            print(f"Но хеши слова '{name}' случайно попали на те единички,")
            print(f"которые до этого оставили Alex, Bob и Charlie.")
            print(f"Текущий массив: {filter.bit_array}")
            break
